# Merge stations and build LOSO folds

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

@dataclass(frozen=True)
class PipelineConfig:

    base_dir: str = str(OUT_DIR)

    expected_stations: tuple = (
        "Islamabad", "Ningxia", "Bolzano", "Helsinki", "Atacama", "Yulara", "Elazig",
    )

    standard_columns: tuple = (
        "utctime", "Power_Actual_W", "Power_PU", "Irradiance_POA",
        "GHI", "Temp_Ambient", "Wind_Speed", "Humidity",
        "HOUR_SIN", "HOUR_COS", "MONTH_SIN", "MONTH_COS",
        "Latitude", "Longitude", "Tilt",
    )

    power_column_aliases: tuple = ("Original_Production_W", "Power_W")

    known_timezone_offsets: dict = field(default_factory=lambda: {
        "Islamabad": 5,
        "Ningxia": 8,
        "Yulara": 9.5,
        "Elazig": 3,
    })

    fallback_rebuild_year: int = 2020

    solar_noon_tolerance_hours: float = 2.0

    ningxia_extra_shift_hours: float = 2.0

    underperformance_quantile: float = 0.10

    nameplate_capacity_w: dict = field(default_factory=lambda: {
        "Islamabad": 1_000.0,
        "Helsinki": 20_000.0,
        "Ningxia": 150_000.0,
        "Yulara": 1_058_400.0,
        "Atacama": 1_408.0,
    })

    power_normalization_percentile: float = 99.0

    eval_split_fraction: float = 0.2
    eval_split_random_seed: int = 42

    loso_folder_name: str = "LOSO_Folds"

    @property
    def train_path(self) -> str:
        return os.path.join(self.base_dir, "Global_Train_Data.csv")

    @property
    def loso_folder_path(self) -> str:
        return os.path.join(self.base_dir, self.loso_folder_name)

    @property
    def eval_train_path(self) -> str:
        return os.path.join(self.base_dir, "Golden_Train_Split.csv")

    @property
    def eval_test_path(self) -> str:
        return os.path.join(self.base_dir, "Golden_Eval_Split.csv")

    @property
    def time_fixed_path(self) -> str:
        return os.path.join(self.base_dir, "Global_Train_Data_TimeFixed.csv")

    @property
    def final_path(self) -> str:
        return os.path.join(self.base_dir, "Global_Train_Data_Time_and_Cyclic_Fixed.csv")

CFG = PipelineConfig()

## Functions

In [ ]:
def discover_station_files(folder: str, station_names: tuple[str, ...]) -> dict[str, str]:
    all_csvs = [f for f in os.listdir(folder) if f.endswith(".csv")]
    stems_lower = {os.path.splitext(f)[0].lower(): f for f in all_csvs}

    resolved: dict[str, str] = {}
    missing: list[str] = []
    for name in station_names:
        exact = f"{name}.csv"
        if exact in all_csvs:
            resolved[name] = os.path.join(folder, exact)
        elif name.lower() in stems_lower:
            resolved[name] = os.path.join(folder, stems_lower[name.lower()])
        else:
            missing.append(name)

    return resolved

In [ ]:
def load_and_harmonize_station(path: str, site_name: str, cfg: PipelineConfig) -> pd.DataFrame | None:
    df = pd.read_csv(path)

    for alias in cfg.power_column_aliases:
        if alias in df.columns:
            df = df.rename(columns={alias: "Power_Actual_W"})
            break

    if "Power_Actual_W" not in df.columns:
        return None

    existing_cols = [c for c in cfg.standard_columns if c in df.columns]
    df = df[existing_cols].copy()

    time_col = "utctime" if "utctime" in df.columns else ("time" if "time" in df.columns else None)
    if time_col:
        df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")

    df["Site_ID"] = site_name
    return df

def build_combined_dataset(folder: str, cfg: PipelineConfig) -> pd.DataFrame:
    station_files = discover_station_files(folder, cfg.expected_stations)

    frames: list[pd.DataFrame] = []
    for site_name, path in station_files.items():
        df = load_and_harmonize_station(path, site_name, cfg)
        if df is not None:
            frames.append(df)

    if not frames:
        raise RuntimeError("No station files found to merge.")

    combined_df = pd.concat(frames, ignore_index=True)
    combined_df.to_csv(cfg.train_path, index=False)

    return combined_df

In [ ]:
def drop_missing_power_rows(df: pd.DataFrame, subset: list[str]) -> pd.DataFrame:
    cleaned = df.dropna(subset=subset).copy()
    return cleaned

def flag_underperforming_days(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:

    df = df.copy()
    df["Underperforming"] = False
    day = df["utctime"].dt.date

    rows = []
    for site in sorted(df["Site_ID"].unique()):
        m = (df["Site_ID"] == site) & (df["Irradiance_POA"] > 50)
        eff = (df.loc[m, "Power_Actual_W"] / df.loc[m, "Irradiance_POA"]).groupby(day[m]).median()
        if eff.empty:
            continue

        thr = eff.quantile(cfg.underperformance_quantile)
        weak = set(eff[eff < thr].index)
        flag = (df["Site_ID"] == site) & day.isin(weak)
        df.loc[flag, "Underperforming"] = True

        keep = m & ~flag
        c_all = df.loc[m, "Irradiance_POA"].corr(df.loc[m, "Power_Actual_W"])
        c_kept = df.loc[keep, "Irradiance_POA"].corr(df.loc[keep, "Power_Actual_W"])

        rows.append({"Site_ID": site, "WeakDays": len(weak),
                     "FlaggedHoursPct": flag.sum() / (df["Site_ID"] == site).sum() * 100,
                     "DailyEffCV": eff.std() / eff.median(),
                     "Corr_All": c_all, "Corr_Kept": c_kept})

    report = pd.DataFrame(rows)
    return df, report

In [ ]:
def repair_broken_timestamps(df: pd.DataFrame, time_col: str, cfg: PipelineConfig) -> pd.DataFrame:
    df = df.copy()
    mask = df[time_col].isnull()
    if mask.sum() == 0:
        return df

    problematic_sites = df.loc[mask, "Site_ID"].unique()

    for site in problematic_sites:
        idx = df[df["Site_ID"] == site].index
        rebuilt = pd.date_range(
            start=f"{cfg.fallback_rebuild_year}-01-01 00:00:00",
            periods=len(idx), freq="h", tz="UTC",
        )
        df.loc[idx, time_col] = rebuilt
    return df

def apply_known_utc_offsets(df: pd.DataFrame, time_col: str, cfg: PipelineConfig) -> pd.DataFrame:
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True)

    for site, offset in cfg.known_timezone_offsets.items():
        mask = df["Site_ID"] == site
        if mask.any():
            df.loc[mask, time_col] = df.loc[mask, time_col] - pd.Timedelta(hours=offset)
    return df

def recompute_cyclic_hour_features(
    df: pd.DataFrame, time_col: str, mask: pd.Series | None = None
) -> pd.DataFrame:
    df = df.copy()
    target_mask = mask if mask is not None else pd.Series(True, index=df.index)

    for site in df.loc[target_mask, "Site_ID"].unique():
        site_mask = target_mask & (df["Site_ID"] == site)
        lon = df.loc[site_mask, "Longitude"].iloc[0]
        local_offset = np.round(lon / 15)

        local_time = df.loc[site_mask, time_col] + pd.Timedelta(hours=local_offset)
        local_hour = local_time.dt.hour + local_time.dt.minute / 60.0

        df.loc[site_mask, "HOUR_SIN"] = np.sin(2 * np.pi * local_hour / 24)
        df.loc[site_mask, "HOUR_COS"] = np.cos(2 * np.pi * local_hour / 24)

    return df

In [ ]:
def apply_station_time_shift(
    df: pd.DataFrame, site_id: str, extra_hours: float, time_col: str
) -> pd.DataFrame:
    df = df.copy()
    mask = df["Site_ID"] == site_id
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df.loc[mask, time_col] = df.loc[mask, time_col] - pd.Timedelta(hours=extra_hours)
    df = recompute_cyclic_hour_features(df, time_col, mask=mask)
    return df

def normalize_power_p99(
    df: pd.DataFrame, site_id: str, percentile: float,
    power_col: str = "Power_Actual_W", pu_col: str = "Power_PU",
) -> tuple[pd.DataFrame, float]:
    df = df.copy()
    mask = df["Site_ID"] == site_id
    if not mask.any():
        return df, float("nan")

    power = df.loc[mask, power_col]
    p_max = np.percentile(power[power > 0], percentile)

    df.loc[mask, pu_col] = (power / p_max).clip(upper=1.0)
    return df, p_max

def normalize_power_by_capacity(
    df: pd.DataFrame, site_id: str, capacity_w: float,
    power_col: str = "Power_Actual_W", pu_col: str = "Power_PU",
) -> pd.DataFrame:
    df = df.copy()
    mask = df["Site_ID"] == site_id
    if not mask.any():
        return df

    power = df.loc[mask, power_col]
    df.loc[mask, pu_col] = (power / capacity_w).clip(upper=1.0)
    return df

def normalize_all_stations(
    df: pd.DataFrame, cfg: PipelineConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    pct = cfg.power_normalization_percentile
    rows = []
    for site in sorted(df["Site_ID"].unique()):
        capacity = cfg.nameplate_capacity_w.get(site)
        if capacity is not None:
            df = normalize_power_by_capacity(df, site_id=site, capacity_w=capacity)
            basis, used = "nameplate capacity", capacity
        else:
            df, used = normalize_power_p99(df, site_id=site, percentile=pct)
            basis = f"percentile (P{pct:.0f})"

        site_mask = df["Site_ID"] == site
        rows.append({
            "Site_ID": site,
            "NormalizationBasis": basis,
            "DenominatorW": used,
            "PU_max_after": df.loc[site_mask, "Power_PU"].max(),
        })

    report = pd.DataFrame(rows)
    return df, report

In [ ]:
def generate_loso_folds(
    df: pd.DataFrame, cfg: PipelineConfig, save_to_disk: bool = True
) -> dict[str, tuple[pd.DataFrame, pd.DataFrame]]:
    sites = sorted(df["Site_ID"].unique())
    if len(sites) < 2:
        raise ValueError("At least two stations are required for LOSO.")

    if save_to_disk:
        os.makedirs(cfg.loso_folder_path, exist_ok=True)

    folds: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
    for site in sites:
        test_fold = df[df["Site_ID"] == site].copy()
        train_fold = df[df["Site_ID"] != site].copy()

        if save_to_disk:
            train_path = os.path.join(cfg.loso_folder_path, f"LOSO_{site}_train.csv")
            test_path = os.path.join(cfg.loso_folder_path, f"LOSO_{site}_test.csv")
            train_fold.to_csv(train_path, index=False)
            test_fold.to_csv(test_path, index=False)

        folds[site] = (train_fold, test_fold)

    return folds

## Run

### Merge stations

In [ ]:
train_df = build_combined_dataset(CFG.base_dir, CFG)

### Drop rows without power

In [ ]:
train_df = drop_missing_power_rows(train_df, subset=["Power_Actual_W", "Power_PU"])
train_df.to_csv(CFG.train_path, index=False)

### Timestamps to UTC

In [ ]:
time_col = "utctime" if "utctime" in train_df.columns else "time"
train_df[time_col] = pd.to_datetime(train_df[time_col], utc=True, errors="coerce")

In [ ]:
train_df = repair_broken_timestamps(train_df, time_col, CFG)
train_df = apply_known_utc_offsets(train_df, time_col, CFG)

train_df.to_csv(CFG.time_fixed_path, index=False)

### Cyclic hour features

In [ ]:
df_final = pd.read_csv(CFG.time_fixed_path)
df_final[time_col] = pd.to_datetime(df_final[time_col], utc=True)

df_final = recompute_cyclic_hour_features(df_final, time_col)

df_final.to_csv(CFG.final_path, index=False)

### Station-specific time shift and normalisation

In [ ]:
df_final = apply_station_time_shift(
    df_final, site_id="Ningxia",
    extra_hours=CFG.ningxia_extra_shift_hours, time_col=time_col,
)

df_final, normalization_report = normalize_all_stations(df_final, CFG)

df_final.to_csv(CFG.final_path, index=False)

### Flag underperforming days

In [ ]:
df_final, _underperf_report = flag_underperforming_days(df_final, CFG)

df_final.to_csv(CFG.final_path, index=False)

### LOSO folds

In [ ]:
loso_folds = generate_loso_folds(df_final, CFG, save_to_disk=True)